# Tome 3, Appendix B: Extremal Problems and Ramsey Theory (Python companion)

Exercises first, solutions at the end. We compute small extremal numbers exhaustively, run Zykov's symmetrisation as an algorithm, build the projective-plane graphs without squares, check the colourings behind $R(3,4)$, $R(3,5)$, $R(4,4)$ and $R(3,3,3)$, prove small Ramsey, Schur and van der Waerden numbers with a SAT solver written from scratch, test Schur's theorem on Fermat's equation modulo primes, and hunt for sunflowers.

*Before each solution, predict what the output will look like.*

In [ ]:
import itertools, random, math, time, sys
from collections import Counter, defaultdict
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
random.seed(8); rng = np.random.default_rng(8)
sys.setrecursionlimit(20000)

## Exercises

**P8.1 (★) Small extremal numbers.** The networkx graph atlas lists all $1253$ graphs with at most $7$ vertices. Compute $\mathrm{ex}(n,K_3)$ and $\mathrm{ex}(n,K_4)$ for $n\le7$, and check that the extremal graphs are exactly the Turán graphs (Theorems 8.1 and 8.4). Do the same for $C_4$ and compare with Theorem 8.6.

**P8.2 (★★) Zykov symmetrisation as an algorithm.** Start from a random $K_{r+1}$-free graph and repeat the two cloning moves of the proof of Theorem 8.4 until no move applies. Check that the number of edges never decreases and that the final graph is complete multipartite with at most $r$ classes. How often does it end at $T(n,r)$ itself?

**P8.3 (★★) Graphs without squares.** Build the polarity graphs $G_p$ of Theorem 8.7 for $p\le13$, check that they contain no $C_4$, count their edges and their self-orthogonal points, and compare with the upper bound of Theorem 8.6. Compare also with the random greedy process that adds random edges as long as no square appears.

**P8.4 (★★) Ramsey colourings and a SAT solver.** Write a small DPLL SAT solver with unit propagation. Encode "$K_n$ has a red–blue colouring with no red $K_k$ and no blue $K_l$" as clauses, and use it to find good colourings of $K_8$ and $K_{13}$ and to prove $R(3,4)\le9$. Check the three circulant colourings of Theorem 8.9, and the colouring of $K_{16}$ of Exercise 8.5 built from the field with $16$ elements.

**P8.5 (★) Schur and Fermat.** Use the solver to find $S(2)=4$ and $S(3)=13$, and Schur's recursive construction to show $S(4)\ge40$. For $m=3,\dots,6$, find all primes $p$ for which $x^m+y^m\equiv z^m\pmod p$ has no solution with $xyz\not\equiv0$ (Corollary 8.14 says it suffices to look below $R_m(3)\le\lfloor e\,m!\rfloor+1$).

**P8.6 (★★) Van der Waerden numbers.** Prove $W(3,2)=9$ and $W(3,3)=27$ with the solver, and find a colouring of $[34]$ in two colours without a monochromatic $4$-AP. Compare with the random lower bound of Exercise 8.11.

**P8.7 (★) How big is colour focusing?** Compute the bound $N(r)$ of Lemma 8.16 for $k=3$ and $r=2$, using $W(2,r')=r'+1$. How many digits does it have?

**P8.8 (★★) Sunflowers.** Find by exhaustive search the largest family of $2$-element sets with no sunflower with $3$ petals, compare with Theorem 8.18 and Exercise 8.12, and run the proof of Theorem 8.18 as an algorithm on random families of $3$-sets.

## Solutions

### P8.1 Small extremal numbers

For each graph of the atlas we test the forbidden subgraph with networkx's subgraph isomorphism (monomorphism, which allows extra edges).

In [ ]:
from networkx.generators.atlas import graph_atlas_g
atlas = graph_atlas_g()
def contains(G, H):
    return nx.algorithms.isomorphism.GraphMatcher(G, H).subgraph_is_monomorphic()
def turan(n, r):
    return nx.complete_multipartite_graph(*[len(range(i, n, r)) for i in range(r)])
for name, H in [('K3', nx.complete_graph(3)), ('K4', nx.complete_graph(4)), ('C4', nx.cycle_graph(4))]:
    row = []
    for n in range(1, 8):
        best, ext = -1, []
        for G in atlas:
            if G.number_of_nodes() != n or G.number_of_edges() < best: continue
            if contains(G, H): continue
            if G.number_of_edges() > best: best, ext = G.number_of_edges(), [G]
            else: ext.append(G)
        row.append((n, best, len(ext)))
        if name != 'C4':
            r = 2 if name == 'K3' else 3
            assert best == turan(n, r).number_of_edges() and len(ext) == 1 and nx.is_isomorphic(ext[0], turan(n, r))
    print(name, 'ex(n) and number of extremal graphs for n=1..7:', [(b, c) for _, b, c in row])
print('ex(n,K3), ex(n,K4) equal t(n,2), t(n,3), with the Turan graph the unique extremal graph: checked')

For $C_4$ the extremal numbers $0,1,3,4,6,7,9$ match the known sequence (OEIS A006855), and the extremal graphs are not unique: the "Turán graph" phenomenon is special to cliques. For $n=7$, Exercise 8.3 says the polarity graph $G_2$ is one of them.

### P8.2 Zykov symmetrisation

A move clones a vertex onto a non-adjacent vertex of smaller degree, or, when $u\not\sim v\not\sim w$ with $u\sim w$ and no single move helps, clones $v$ onto both $u$ and $w$. The proof shows that a graph where no move strictly helps is complete multipartite; here we apply moves whenever they do not decrease the edge count and strictly simplify, and stop when non-adjacency is an equivalence relation.

In [ ]:
def has_clique(G, k):
    return any(len(c) >= k for c in nx.find_cliques(G))
def random_free(n, r, tries=4000):
    G = nx.empty_graph(n)
    E = list(itertools.combinations(range(n), 2)); random.shuffle(E)
    for (a, b) in E[:tries]:
        G.add_edge(a, b)
        if has_clique(G, r + 1): G.remove_edge(a, b)
    return G
def clone(G, u, v):            # make u a twin of v (u, v non-adjacent)
    G.remove_edges_from(list(G.edges(u)))
    G.add_edges_from((u, x) for x in list(G[v]))
def zykov(G):
    G = G.copy(); hist = [G.number_of_edges()]
    while True:
        n = G.number_of_nodes(); moved = False
        for u, v, w in itertools.permutations(range(n), 3):
            if not G.has_edge(u, v) and not G.has_edge(v, w) and G.has_edge(u, w):
                if G.degree(u) > G.degree(v): clone(G, v, u)
                elif G.degree(w) > G.degree(v): clone(G, v, w)
                else: clone(G, u, v); clone(G, w, v)
                hist.append(G.number_of_edges()); moved = True; break
        if not moved: return G, hist
def classes(G):
    n = G.number_of_nodes(); cls = []
    for v in range(n):
        for c in cls:
            if not G.has_edge(v, c[0]): c.append(v); break
        else: cls.append([v])
    return sorted(len(c) for c in cls)
stats = Counter()
for trial in range(40):
    n, r = random.randint(6, 12), random.randint(2, 4)
    G0 = random_free(n, r)
    G, hist = zykov(G0)
    assert all(a <= b for a, b in zip(hist, hist[1:])), hist
    assert not has_clique(G, r + 1)
    cl = classes(G)
    # complete multipartite: edges = (n^2 - sum sizes^2)/2
    assert G.number_of_edges() == (n * n - sum(s * s for s in cl)) // 2 and len(cl) <= r
    stats['ends at T(n,r)' if G.number_of_edges() == turan(n, r).number_of_edges() else 'complete multipartite, fewer edges'] += 1
print(stats)
print('example run, edge counts:', hist, ' final class sizes', cl)

The moves always end at a complete multipartite graph, but not always at the Turán graph: the cloning moves never *balance* the classes, which is the job of Lemma 8.3. The proof of Theorem 8.4 starts from a graph with the most edges, where no move can strictly increase the count.

### P8.3 Graphs without squares

In [ ]:
def proj_points(p):
    pts = []
    for v in itertools.product(range(p), repeat=3):
        if any(v):
            i = next(j for j in range(3) if v[j])
            if v[i] == 1: pts.append(v)          # normalised: first non-zero coordinate is 1
    return pts
def polarity(p):
    P = proj_points(p); G = nx.Graph(); G.add_nodes_from(range(len(P)))
    for i, j in itertools.combinations(range(len(P)), 2):
        if sum(a * b for a, b in zip(P[i], P[j])) % p == 0: G.add_edge(i, j)
    absolute = sum(1 for x in P if sum(a * a for a in x) % p == 0)
    return G, absolute
def has_c4(G):
    for u, v in itertools.combinations(G.nodes, 2):
        if len(set(G[u]) & set(G[v])) >= 2: return True
    return False
def kst(n): return 0.5 * n ** 1.5 + 0.5 * n
def greedy_c4free(n):
    G = nx.empty_graph(n); E = list(itertools.combinations(range(n), 2)); random.shuffle(E)
    for a, b in E:
        # adding ab creates a square iff a and b are joined by a path of length 3
        if any(G.has_edge(x, y) for x in G[a] for y in G[b] if x != b and y != a and x != y): continue
        G.add_edge(a, b)
    assert not has_c4(G); return G
print(' p    n   edges  absolute     pn/2  KST bound  n^1.5/2  greedy')
for p in [2, 3, 5, 7, 11, 13]:
    G, a = polarity(p); n = G.number_of_nodes()
    assert not has_c4(G) and n == p * p + p + 1
    assert a == p + 1                                          # a conic has p+1 points (p+1 = 3 for p=2 too)
    assert G.number_of_edges() == ((p + 1) * n - a) // 2       # degree p+1, or p at the absolute points
    print(f'{p:2d} {n:4d} {G.number_of_edges():6d} {a:6d} {p*n/2:12.0f} {kst(n):9.0f} {0.5*n**1.5:8.0f} {greedy_c4free(n).number_of_edges():7d}')

Each graph has exactly $p+1$ self-orthogonal points (a conic, or a line for $p=2$), so the number of edges is $\frac12\bigl((p+1)n-(p+1)\bigr)=\frac12(p+1)(p^2+p)$, slightly above the bound $\frac12pn$ proved in the text, and remarkably close to the Kővári–Sós–Turán bound. The random greedy process stops far below: a random-looking graph wastes edges, and the algebraic one wastes almost none. (Füredi proved in 1996 that for $q$ a large prime power, $\mathrm{ex}(q^2+q+1,C_4)=\frac12q(q+1)^2$ exactly: the polarity graphs are optimal.)

### P8.4 Ramsey colourings and a SAT solver

The solver below is DPLL (Davis–Putnam–Logemann–Loveland, 1962) with the *two watched literals* trick: each clause watches two of its literals, and is only examined when one of them becomes false. It branches on the most frequent variable first, tries both values, and undoes its assignments on failure. Variables are numbered from $1$; a literal is $\pm$ a variable.

In [ ]:
def dpll(nv, clauses):
    cl = [list(dict.fromkeys(c)) for c in clauses]
    watch = defaultdict(list)
    for i, c in enumerate(cl):
        for l in c[:2]: watch[l].append(i)
    val, trail = {}, []
    def value(l):
        v = val.get(abs(l)); return None if v is None else (v if l > 0 else not v)
    def assign(l): val[abs(l)] = l > 0; trail.append(abs(l))
    def propagate(q):
        while q:
            fl = -q.pop(); ws = watch[fl]; i = 0
            while i < len(ws):
                c = cl[ws[i]]
                if len(c) == 1:
                    if value(c[0]) is False: return False
                    i += 1; continue
                if c[0] == fl: c[0], c[1] = c[1], c[0]
                if value(c[0]) is True: i += 1; continue
                for k in range(2, len(c)):
                    if value(c[k]) is not False:
                        c[1], c[k] = c[k], c[1]; watch[c[1]].append(ws[i]); ws[i] = ws[-1]; ws.pop(); break
                else:
                    v0 = value(c[0])
                    if v0 is False: return False
                    if v0 is None: assign(c[0]); q.append(c[0])
                    i += 1
        return True
    units = [c[0] for c in cl if len(c) == 1]
    for u in units:
        if value(u) is False: return None
        if value(u) is None: assign(u)
    if not propagate(list(units)): return None
    occ = Counter(abs(l) for c in cl for l in c)
    order = sorted(range(1, nv + 1), key=lambda v: -occ[v])
    def rec():
        v = next((v for v in order if v not in val), None)
        if v is None: return True
        for lit in (v, -v):
            mark = len(trail); assign(lit)
            if propagate([lit]) and rec(): return True
            while len(trail) > mark: del val[trail.pop()]
        return False
    return dict(val) if rec() else None
def check_model(model, clauses):
    return all(any(model.get(abs(l), False) == (l > 0) for l in c) for c in clauses)

def ramsey_cnf(n, k, l):
    idx = {e: i + 1 for i, e in enumerate(itertools.combinations(range(n), 2))}
    cls = [[-idx[e] for e in itertools.combinations(S, 2)] for S in itertools.combinations(range(n), k)]   # no red K_k
    cls += [[idx[e] for e in itertools.combinations(S, 2)] for S in itertools.combinations(range(n), l)]   # no blue K_l
    return idx, cls
for n, k, l in [(5, 3, 3), (6, 3, 3), (8, 3, 4), (9, 3, 4), (13, 3, 5)]:
    t = time.time(); idx, cls = ramsey_cnf(n, k, l); m = dpll(len(idx), cls)
    if m is not None: assert check_model(m, cls)
    print(f'K_{n}: good colouring for R({k},{l})? {"yes" if m else "no "}   ({len(cls)} clauses, {time.time()-t:.2f} s)')

So $R(3,3)=6$ and $R(3,4)=9$ are proved by the machine as well, the "no" answers being exhaustive searches. (Proving $R(3,5)\le14$ or $R(4,4)\le18$ this way is beyond such a small solver; the text needs no search for them.) Now the three circulant colourings, checked by brute force over all $k$- and $l$-sets, and the Paley graph's self-complementarity.

In [ ]:
def circulant(n, S):
    return nx.Graph([(i, j) for i, j in itertools.combinations(range(n), 2) if (j - i) % n in S])
def good(G, n, k, l):
    red = lambda S: all(G.has_edge(a, b) for a, b in itertools.combinations(S, 2))
    blue = lambda S: not any(G.has_edge(a, b) for a, b in itertools.combinations(S, 2))
    return not any(red(S) for S in itertools.combinations(range(n), k)) and not any(blue(S) for S in itertools.combinations(range(n), l))
Q17 = {x * x % 17 for x in range(1, 17)}
for n, S, k, l in [(8, {1, 4, 7}, 3, 4), (13, {1, 5, 8, 12}, 3, 5), (17, Q17, 4, 4)]:
    G = circulant(n, S); print(f'Z/{n}, S={sorted(S)}: no red K{k}, no blue K{l}:', good(G, n, k, l))
P17 = circulant(17, Q17)
print('P17 is self-complementary via x -> 3x:', all(P17.has_edge(3 * a % 17, 3 * b % 17) != P17.has_edge(a, b) for a, b in itertools.combinations(range(17), 2)))
print('cubes mod 13:', sorted({x ** 3 % 13 for x in range(1, 13)}))

The field with $16$ elements is $\mathbb F_2[x]/(x^4+x+1)$: elements are $4$-bit integers, addition is XOR, multiplication is carry-less multiplication reduced modulo $x^4+x+1$ (binary `10011`).

In [ ]:
def gf16_mul(a, b):
    r = 0
    for i in range(4):
        if b >> i & 1: r ^= a << i
    for i in (6, 5, 4):
        if r >> i & 1: r ^= 0b10011 << (i - 4)
    return r
def gf16_pow(a, e):
    r = 1
    for _ in range(e): r = gf16_mul(r, a)
    return r
g = next(a for a in range(2, 16) if len({gf16_pow(a, e) for e in range(15)}) == 15)   # a generator
H = {gf16_pow(g, 3 * e) for e in range(5)}                                           # the cubes
coset = {}
for c in range(3):
    for h in H: coset[gf16_mul(gf16_pow(g, c), h)] = c
assert len(coset) == 15
colour = lambda x, y: coset[x ^ y]
mono = [T for T in itertools.combinations(range(16), 3) if len({colour(T[0], T[1]), colour(T[1], T[2]), colour(T[0], T[2])}) == 1]
print('generator', g, ' cubes H =', sorted(H), ' monochromatic triangles:', len(mono))
for c in range(3):
    Gc = nx.Graph([(x, y) for x, y in itertools.combinations(range(16), 2) if colour(x, y) == c])
    deg = {d for _, d in Gc.degree()}
    lam = {len(set(Gc[x]) & set(Gc[y])) for x, y in Gc.edges}
    mu = {len(set(Gc[x]) & set(Gc[y])) for x, y in itertools.combinations(range(16), 2) if not Gc.has_edge(x, y)}
    print(f'colour {c}: degrees {deg}, common neighbours of adjacent {lam}, of non-adjacent {mu}  (Clebsch graph: 5, 0, 2)')

Each colour class is strongly regular with parameters $(16,5,0,2)$: the Clebsch graph. Hence $R(3,3,3)=17$, which the solver could also find by itself (try it with three colours: it is a bigger instance).

### P8.5 Schur and Fermat

In [ ]:
def sum_free(P):
    s = set(P); return not any(x + y in s for x in P for y in P)
def schur_cnf(N, r):
    var = lambda i, c: (i - 1) * r + c + 1
    cls = [[var(i, c) for c in range(r)] for i in range(1, N + 1)]
    for x in range(1, N + 1):
        for y in range(x, N + 1 - x):
            for c in range(r): cls.append([-var(x, c), -var(y, c), -var(x + y, c)])
    cls.append([var(1, 0)])                                  # symmetry: 1 gets colour 0
    return cls, var
for r, N in [(2, 4), (2, 5), (3, 13), (3, 14)]:
    cls, var = schur_cnf(N, r); m = dpll(N * r, cls)
    if m:
        col = {i: next(c for c in range(r) if m.get(var(i, c))) for i in range(1, N + 1)}   # first true colour
        parts = [[i for i in range(1, N + 1) if col[i] == c] for c in range(r)]
        assert all(map(sum_free, parts))
        print(f'[{N}] into {r} sum-free sets: {parts}')
    else:
        print(f'[{N}] cannot be split into {r} sum-free sets')
# Schur's construction: from a partition of [n] into r sum-free sets, one of [3n+1] into r+1.
def schur_up(parts, n):
    new = [sorted(set(P) | {x + 2 * n + 1 for x in P}) for P in parts]
    new.append(list(range(n + 1, 2 * n + 2)))
    return new, 3 * n + 1
parts, n = [[1]], 1
for r in range(2, 6):
    parts, n = schur_up(parts, n)
    assert sorted(x for P in parts for x in P) == list(range(1, n + 1)) and all(map(sum_free, parts))
    print(f'r={r}: [{n}] split into {r} sum-free sets, so S({r}) >= {n}')

Schur's construction is a tiny induction: keep each old set $P$ and add its translate $P+2n+1$, and put the middle block $n+1,\dots,2n+1$ in a new set. The middle block is sum-free because two of its elements add up to at least $2n+2$; in $P\cup(P+2n+1)$, a sum $x+y$ of two old elements is at most $2n$ and not in $P$, a sum $x+(y+2n+1)$ lies in $P+2n+1$ only if $x+y\in P$, and two translated elements add up to more than $3n+1$. It gives $S(r)\ge\frac{3^r-1}2$, so $S(4)\ge40$ and $S(5)\ge121$, to compare with the true $44$ and $160$.

Fermat modulo $p$: by Corollary 8.14, a prime with no solution satisfies $p\le R_m(3)\le\lfloor e\,m!\rfloor+1$, so a finite search finds them all.

In [ ]:
from sympy import primerange
for m in range(3, 7):
    bound = math.floor(math.e * math.factorial(m)) + 1
    bad = []
    for p in primerange(3, bound + 1):
        powers = {pow(x, m, p) for x in range(1, p)}
        if not any((a + b) % p in powers for a in powers for b in powers): bad.append(p)
    print(f'm={m}: primes up to {bound} with no non-trivial solution of x^m+y^m=z^m: {bad}')

For $m=3$ only $7$ and $13$, as Exercise 8.7 claims; the list is complete, because Schur's corollary rules out every $p>17$. Note that $p\equiv2\pmod3$ never appears for $m=3$: then every residue is a cube, and $1+1=2$ is a solution.

### P8.6 Van der Waerden numbers

In [ ]:
def vdw_cnf(N, k, r):
    var = lambda i, c: i * r + c + 1
    cls = [[var(i, c) for c in range(r)] for i in range(N)]
    for a in range(N):
        for d in range(1, N):
            if a + (k - 1) * d >= N: break
            for c in range(r): cls.append([-var(a + j * d, c) for j in range(k)])
    cls.append([var(0, 0)])
    return cls, var
for N, k, r in [(8, 3, 2), (9, 3, 2), (26, 3, 3), (27, 3, 3), (34, 4, 2)]:
    t = time.time(); cls, var = vdw_cnf(N, k, r); m = dpll(N * r, cls)
    if m:
        word = ''.join('RBG'[next(c for c in range(r) if m.get(var(i, c)))] for i in range(N))
        print(f'[{N}], {r} colours, no monochromatic {k}-AP: {word}   ({time.time()-t:.2f} s)')
    else:
        print(f'every {r}-colouring of [{N}] has a monochromatic {k}-AP   ({time.time()-t:.2f} s)')
for k in range(3, 9):
    print(f'k={k}: random bound W(k,2) > {math.floor(math.sqrt(k-1) * 2 ** (k/2))}', ' true value', {3: 9, 4: 35, 5: 178, 6: 1132}.get(k, 'unknown'))

$W(3,3)=27$ takes the little solver a fraction of a second; $W(4,2)=35$ is within reach of a real CDCL solver but takes this one minutes, so we only exhibit the colouring of $[34]$. The random bound is weak but honest: $W(5,2)=178$ against $11$.

### P8.7 How big is colour focusing?

With $k=3$: $N(1)=2W(2,r)=2(r+1)$ and $N(s+1)=N(s)\cdot2W(2,r^{N(s)})=N(s)\cdot2(r^{N(s)}+1)$. For $r=2$ the numbers are small; for $r=3$ we can only count the digits.

In [ ]:
r = 2
N1 = 2 * (r + 1)
N2 = N1 * 2 * (r ** N1 + 1)
print('N(1) =', N1, '  N(2) =', N2, '  so the proof gives W(3,2) <=', N2, ' while W(3,2) = 9')
r = 3
N = 2 * (r + 1); Ns = [N]
N = N * 2 * (r ** N + 1); Ns.append(N)                               # N(2); N(3) is too big to write down
digits = math.floor(math.log10(Ns[1]) + math.log10(2) + Ns[1] * math.log10(3)) + 1   # N(3) = N(2) * 2 * (3^N(2) + 1)
print('k=3, r=3: N(1), N(2) =', Ns[0], Ns[1], ' and N(3) has', digits, 'digits, while W(3,3) = 27')

For $k=4$ one would need $W(3,r^{N})$ with $N$ in the hundreds, and the bound becomes a tower: the gap between what colour focusing proves and the truth is as large as gaps get in mathematics.

### P8.8 Sunflowers

A family of $2$-sets is a graph. A sunflower with $3$ petals is either $3$ disjoint edges (empty core) or $3$ edges at one vertex (core a vertex). So we want the largest graph with matching number at most $2$ and maximum degree at most $2$. Such a graph is a union of paths and cycles. A component with matching number $1$ has at most $3$ edges (a triangle or a path of length at most $2$), and one with matching number $2$ at most $5$ (the cycle $C_5$); so the answer uses at most $6$ vertices, and searching $n\le7$ is enough.

In [ ]:
def has_sunflower(F, p):
    F = [frozenset(A) for A in F]
    for S in itertools.combinations(F, p):
        core = frozenset.intersection(*S)
        if all(a & b == core for a, b in itertools.combinations(S, 2)): return True
    return False
best = 0
for n in range(3, 8):
    E = list(itertools.combinations(range(n), 2))
    # search edge sets with max degree <= 2 by increasing size (cheap pruning by degree)
    def grow(i, chosen, deg):
        global best
        if len(chosen) > best and not has_sunflower(chosen, 3):
            best = len(chosen); print('n =', n, ': family of', best, 'edges without a 3-sunflower:', chosen)
        for j in range(i, len(E)):
            a, b = E[j]
            if deg[a] < 2 and deg[b] < 2:
                deg[a] += 1; deg[b] += 1; chosen.append(E[j])
                if not has_sunflower(chosen, 3): grow(j + 1, chosen, deg)
                chosen.pop(); deg[a] -= 1; deg[b] -= 1
    grow(0, [], Counter())
print('largest:', best, '  Erdos-Rado bound k!(p-1)^k =', 2 * 4, '  sharpness construction (p-1)^k =', 4)

The answer is $6$, two disjoint triangles: between the construction's $4$ and the theorem's $8$. (In general, for $k=2$ the exact answer is known; for $k\ge3$ it is not.) Now the proof as an algorithm: take a maximal disjoint subfamily; if it is too small, find the most popular element of its union and recurse on the sets through it.

In [ ]:
def find_sunflower(F, p):
    F = [frozenset(A) for A in F]
    D, U = [], set()
    for A in F:
        if not (A & U): D.append(A); U |= A
    if len(D) >= p: return D[:p], frozenset()
    x = Counter(e for A in F for e in A & U).most_common(1)[0][0]
    G = [A - {x} for A in F if x in A]
    if len(set(G)) < len(G) or not all(G): return None
    res = find_sunflower(G, p)
    if res is None: return None
    S, core = res
    return [A | {x} for A in S], core | {x}
k, p = 3, 3
found = 0
for trial in range(200):
    size = math.factorial(k) * (p - 1) ** k + 1          # 49 sets: the theorem guarantees a sunflower
    F = set()
    while len(F) < size: F.add(frozenset(random.sample(range(12), k)))
    res = find_sunflower(list(F), p)
    assert res is not None
    S, core = res
    assert all(a & b == core for a, b in itertools.combinations(S, 2)) and len(set(S)) == p
    found += 1
print(found, 'random families of', size, 'three-element sets: the proof always found a sunflower with', p, 'petals')
sharp = [frozenset((i, a) for i, a in enumerate(t)) for t in itertools.product(range(p - 1), repeat=k)]
print('the (p-1)^k =', len(sharp), 'sets of Exercise 8.12 contain a sunflower?', has_sunflower(sharp, p))